# Модель B: реранкер кандидатов/чанков (Yandex Cloud, plan §6.4, §7)

cross-encoder `BAAI/bge-reranker-v2-m3`. В пайплайне инференс — `src/adapters/rerank/bge_reranker.py` (`RerankerPort`): ранжирует RAG-чанки полного текста и `title+abstract` кандидатов по релевантности PICO.

Адаптер грузит HF-модель напрямую (чекпойнт не обязателен). Этот ноутбук: (1) оценка **nDCG@10** на gold query->relevance; (2) опциональный fine-tune CrossEncoder на доменных парах с сохранением локального чекпойнта (тогда `model_name` адаптера = путь).

In [ ]:
# %pip install -q sentence-transformers scikit-learn

In [ ]:
from sentence_transformers import CrossEncoder

model = CrossEncoder("BAAI/bge-reranker-v2-m3")

# Gold: запрос + пассажи с бинарной релевантностью (1 — целевой pivotal-РКИ).
query = "melanoma dabrafenib trametinib overall survival hazard ratio"
passages = [
    ("COMBI-d: dabrafenib+trametinib, OS hazard ratio for death 0.71.", 1),
    ("Narrative review of BRAF pathway in melanoma.", 0),
    ("coBRIM: vemurafenib+cobimetinib, overall survival benefit.", 1),
    ("Cost analysis of melanoma therapies.", 0),
]
scores = model.predict([(query, p) for p, _ in passages])
list(zip([p for p, _ in passages], scores, strict=False))

In [ ]:
import numpy as np


def ndcg_at_k(rels, k=10):
    rels = np.asarray(rels)[:k]
    if rels.size == 0:
        return 0.0
    discounts = 1.0 / np.log2(np.arange(2, rels.size + 2))
    dcg = float((rels * discounts).sum())
    ideal = np.sort(rels)[::-1]
    idcg = float((ideal * discounts).sum())
    return dcg / idcg if idcg else 0.0


order = np.argsort(scores)[::-1]
ranked_rel = [passages[i][1] for i in order]
print("nDCG@10:", ndcg_at_k(ranked_rel, k=10))

In [ ]:
# Опциональный fine-tune (если нужен доменный чекпойнт):
# from sentence_transformers import InputExample
# from torch.utils.data import DataLoader
# train = [InputExample(texts=[query, p], label=float(r)) for p, r in passages]
# model.fit(DataLoader(train, batch_size=2), epochs=1, warmup_steps=0)
# model.save('bge_reranker_ft')  # -> Object Storage; model_name адаптера = этот путь